## U ovom delu analiziramo sledece, kako je prof rekao:

1. Albon, C. (2018). Machine Learning with Python Cookbook: Practical Solutions from Preprocessing to Deep Learning. O’Reilly Media. Poglavlja 2 i 3 obrađuju učitavanje podataka i rada sa tabelama. Za ovu temu najvažnija su poglavlja 4 i 5, posvećena numeričkim i kategoričkim podacima.

2. scikit-learn dokumentacija. Posebno su značajni odeljci o standardizaciji, skaliranju na interval i kodiranju kategoričkih atributa, a nakon toga primer upotrebe klase SimpleImputer. 

## 1. Priprema okruženja i podela podataka (Train/Test Split)

**Šta radimo ovde?**
Kreiramo veštačku tabelu sa podacima i odmah je delimo na dva dela: skup za obučavanje (`X_train`) i skup za testiranje (`X_test`).

**Zašto to radimo?**
U mašinskom učenju uvek moramo odvojiti test podatke na samom početku. Sve pripreme i računanja (prosek, medijana, skaliranje) vršimo **isključivo** na trening skupu da test podaci ne bi "procureli" u model (**Data Leakage**).

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder

# Sintetički skup podataka za demonstraciju
data = {
    'Starost': [25, 30, np.nan, 45, 22, 110],  # sadrži nedostajuću vrednost i outlier (110)
    'Plata': [50000, 60000, 55000, 120000, 48000, 52000],
    'Grad': ['Beograd', 'Nis', 'Novi Sad', 'Beograd', 'Nis', 'Nis'], # nominalni
    'Obrazovanje': ['Osnovno', 'Master', 'Srednje', 'Doktorat', 'Srednje', 'Master'] # ordinalni
}
df = pd.DataFrame(data)

## 2. Popunjavanje nedostajućih vrednosti (Imputacija)

**Šta radimo ovde?**
Imamo rupu u podacima (`NaN` kod Starosti). Koristimo `SimpleImputer` sa strategijom `median` (medijana) da popunimo tu prazninu.

**Zašto medijana, a ne prosek (srednja vrednost)?**
U koloni `Starost` imamo vrednost 110 (ekstremna vrednost / outlier). Prosek bi zbog nje ispao nerealan (npr. 46 godina), dok medijana traži srednji element u nizu i ostaje stabilna i realna.

**Kako sprečavamo Data Leakage?**
- `imputer.fit(X_train)` -> Računa medijanu SAMO na trening skupu.
- `imputer.transform(...)` -> Popunjava rupe u trening i test skupu pomoću te izračunate medijane.

In [5]:
# Podela na trening i test skup (Zlatno pravilo - sprečavanje Data Leakage-a!)
X_train, X_test = train_test_split(df, test_size=0.3, random_state=42)

# Popunjavanje nedostajućih vrednosti medijanom za numeričke atribute
imputer = SimpleImputer(strategy='median')

# Fitujemo ISKLJUČIVO na trening skupu, pa transformišemo oba
X_train[['Starost']] = imputer.fit_transform(X_train[['Starost']])
X_test[['Starost']] = imputer.transform(X_test[['Starost']])

## 3. Standardizacija numeričkih atributa

**Šta radimo ovde?**
Prevaramo vrednosti u kolonama `Starost` i `Plata` u z-score formate (gde je prosek 0, a odstupanja u standardnim devijacijama).

**Zašto to radimo?**
`Plata` se meri u desetinama hiljada (50,000), a `Starost` u desetinama (25). Algoritmi mašinskog učenja bi mislili da je `Plata` 1000 puta važnija samo zato što su brojevi veći! Skaliranjem ih dovodimo na isti nivo značajnosti.

In [ ]:
scaler = StandardScaler()

# Standardizacija (z-score scaling): mean=0, std=1
X_train[['Starost', 'Plata']] = scaler.fit_transform(X_train[['Starost', 'Plata']])
X_test[['Starost', 'Plata']] = scaler.transform(X_test[['Starost', 'Plata']])

## 4. Kodiranje kategoričkih atributa (Tekst u Brojeve)

**Šta radimo ovde?**
Algoritmi mašinskog učenja znaju da rade samo sa brojevima. Zato tekstualne kolone moramo pretvoriti u brojeve na dva načina:

1. **Ordinal Encoding (za `Obrazovanje`):** Postoji redosled (Osnovno < Srednje < Master < Doktorat). Dodeljujemo im redom brojeve 0, 1, 2, 3.
2. **One-Hot Encoding (za `Grad`):** Gradovi nemaju redosled (Niš nije "veći" ili "manji" od Beograda). Za svaki grad pravimo novu kolonu sa 0 ili 1 (tzv. binarne indikatorske kolone).

In [ ]:
# 4. Ordinalno kodiranje za 'Obrazovanje' (ima redosled)
nivoi = ['Osnovno', 'Srednje', 'Master', 'Doktorat']
ordinal_enc = OrdinalEncoder(categories=[nivoi])

X_train['Obrazovanje_Kod'] = ordinal_enc.fit_transform(X_train[['Obrazovanje']])
X_test['Obrazovanje_Kod'] = ordinal_enc.transform(X_test[['Obrazovanje']])

# 2. One-Hot kodiranje za 'Grad' (nema redosled)
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
grad_ohe_train = ohe.fit_transform(X_train[['Grad']])
grad_ohe_test = ohe.transform(X_test[['Grad']])

## 5. Kreiranje izvedenih atributa (Feature Engineering)

**Šta radimo ovde?**
Iz izvornog kolonskog tipa "Datum i vreme" izvlačimo pojedinačne brojeve: dan u nedelji (0-6) i sat u danu (0-23).

**Zašto to radimo?**
Algoritmi mašinskog učenja ne razumeju tekstualni datum poput `"2026-11-01"`. Međutim, ako izvučemo informaciju da je to bio nedelja (dan 6) ili da je polazak bio u 14 sati, model iz tih brojeva može lakše da prepozna obrasce (npr. da su gužve veće vikendom ili u špicu).

In [9]:
# Пример издвајања temporalnih atributa iz datuma
df_dates = pd.DataFrame({'Datum_Polaska': pd.date_range(start='2026-11-01', periods=3, freq='D')})

df_dates['Dan_U_Nedelji'] = df_dates['Datum_Polaska'].dt.dayofweek
df_dates['Sat_Polaska'] = df_dates['Datum_Polaska'].dt.hour

## 6. Provera i ispis pripremljenih podataka
Ispisujemo finalne rezultate da vidimo kako podaci izgledaju nakon svih primenjenih transformacija.

In [10]:
# Prikaz očiglednog rezultata transformacije nad trening skupom
print("--- Očišćeni i skalirani trening podaci ---")
print(X_train)

print("\n--- One-Hot kodirani gradovi ---")
print(pd.DataFrame(grad_ohe_train, columns=ohe.get_feature_names_out(['Grad'])))

print("\n--- Izvedeni atributi iz datuma ---")
print(df_dates)

--- Očišćeni i skalirani trening podaci ---
    Starost     Plata      Grad Obrazovanje  Obrazovanje_Kod
5  1.659727 -0.564102       Nis      Master              2.0
2 -0.319764 -0.463069  Novi Sad     Srednje              1.0
4 -1.020199 -0.698813       Nis     Srednje              1.0
3 -0.319764  1.725983   Beograd    Doktorat              3.0

--- One-Hot kodirani gradovi ---
   Grad_Beograd  Grad_Nis  Grad_Novi Sad
0           0.0       1.0            0.0
1           0.0       0.0            1.0
2           0.0       1.0            0.0
3           1.0       0.0            0.0

--- Izvedeni atributi iz datuma ---
  Datum_Polaska  Dan_U_Nedelji  Sat_Polaska
0    2026-11-01              6            0
1    2026-11-02              0            0
2    2026-11-03              1            0


## 7. Zaključak

Kroz ovaj primer uspešno smo primenili celokupan celi tok pripreme podataka (**Data Preprocessing**):

1. **Sprečavanje Data Leakage-a:** Podatke smo prvo podelili na trening (`X_train`) i test (`X_test`) skup. Sve parametre transformacije izračunavali smo **isključivo** na trening skupu.
2. **Imputacija (`SimpleImputer`):** Nedostajuću vrednost u koloni `Starost` popunili smo medijanom, čime smo izbegli uticaj ekstremnih vrednosti (outliera).
3. **Standardizacija (`StandardScaler`):** Numeričke atribute (`Starost` i `Plata`) doveli smo na istu skalu sa prosekom 0 i standardnom devijacijom 1, čime sprečavamo da dominantne numeričke vrednosti zavaraju algoritam.
4. **Kodiranje kategorija:** 
   - Ordinalne podatke (`Obrazovanje`) kodirali smo sa `OrdinalEncoder` jer imaju prirodan redosled.
   - Nominalne podatke (`Grad`) kodirali smo sa `OneHotEncoder` i kreirali binarne indikatorske kolone bez rangiranja.
5. **Feature Engineering:** Iz temporalnih podataka (`Datum_Polaska`) izvukli smo izvedene atribute (`Dan_U_Nedelji`, `Sat_Polaska`) pogodne za mašinsko učenje.

Ovako pripremljeni podaci su numerički stabilni, bez praznina i spremni za obučavanje algoritama mašinskog učenja.